In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/sample_emails_with_triage_200.csv")
df.head()

,id,sender,subject,body,priority,triage_label,ideal_intent,ideal_tone
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human,respond,polite
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond,respond,polite
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore,respond,polite
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond,respond,polite
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond,respond,polite


In [4]:
def email_assistant(email_text):
    text = str(email_text).lower()

    if any(k in text for k in ["urgent", "deadline", "submit", "due", "payment", "invoice", "overdue"]):
        return "respond", "urgent"
    if any(k in text for k in ["suspicious", "security alert", "account suspended", "login attempts"]):
        return "notify", "urgent"
    if any(k in text for k in ["sale", "offer", "discount", "congratulations", "winner"]):
        return "ignore", "neutral"
    return "respond", "neutral"

In [5]:
# Define Dangerous Actions
dangerous_actions=["respond"]

In [6]:
# HITL checkpoint logic
def hitl_check(action):
    if action in dangerous_actions:
        return "WAIT_FOR_HUMAN"
    return "AUTO_APPROVE"  

In [7]:
# Stimulate Human Approval
def human_decision():
    decision= input("Approve action? (yes/no): ")
    return decision.lower() == "yes"

In [10]:
results = []
for _, row in df.sample(5).iterrows():
    action, tone=email_assistant(row["body"])

    status= hitl_check(action)
    if status == "WAIT_FOR_HUMAN":
        approved= human_decision()
        final_action= action if approved else "blocked"
    else:
        final_action= action
    results.append({
        "email": row["body"][:50],
        "ai_action": action,
        "final_action": final_action,
        "hitl_status": status
    })

results_df = pd.DataFrame(results)
results_df

,email,ai_action,final_action,hitl_status
0,Please complete the mandatory training module ...,respond,respond,WAIT_FOR_HUMAN
1,Security alert: multiple failed login attempts...,notify,notify,AUTO_APPROVE
2,"Hi, don't miss our sale with discounts up to 7...",ignore,ignore,AUTO_APPROVE
3,Notice: Your account will be locked unless ver...,respond,blocked,WAIT_FOR_HUMAN
4,"Dear user, we detected a login from a new devi...",respond,respond,WAIT_FOR_HUMAN
